In [315]:
# pobieramy nerzędzia i biblioteki

from dotenv import load_dotenv
from openai import OpenAI
from tavily import TavilyClient
from bs4 import BeautifulSoup
from pydantic import BaseModel
import requests
import os
import json
import csv


In [316]:
# ładujemy plik env dzieki load dotenv

load_dotenv(override=True)

# pobiermay klienta openai

openai = OpenAI()

# pobieramy klucz api do narzedzia tavily które robi search po www, taivily pozwala połaczyć nasz llm z internetem  pobieramy klucz z .env metoa os.getenv

tavily = TavilyClient(api_key= os.getenv("TVLY_API_KEY"))

In [317]:
# tworzymy 1 funkcje tool do scrapowania strony, jaką podamy 
# jako parametr podajemy url 




def scrape_website(url):

    # do zmiennej response przekazujemy cała strone html pobraną przez metode request
    # zmetodą get jako parametr podajemy url 
    response = requests.get(url)

    # teraz mamy htlm w response dzieki metodzie requests.get
    # dzieki biblioteece BeautifulSoup przekazujemy pobrany content z response i dodajemy metode html.parser, któa przekłada html na pythona

    soup = BeautifulSoup(response.content, 'html.parser')

    # teraz z soup gdzie mamy cały content z html dzieki response.content przeknowertowany na pythona dzieki html.parsed
    # wyciagamy z niego text i returnem zwracamy do nazwy funkcji

    return soup.get_text()

    


In [318]:
# 2 funkcja która ma wysyła zapytanie do Tavily i zwraca dopowiedzi
# jako parametr podajemy query

def search_similar_companies(query):

    # do zmiennej response wrzucamy obiekt tavily, dodajemy jeko metodę search
    # i 2 zmienne wymagane przez narzędzie api query i max_results

    response = tavily.search(query=query, max_results=5)

    # z response wyciagy liste wyników

    return response['results']

In [319]:
# 3 funkcja która zapisuje prospekt do csv przyjmue jako parametr company,url, reasin i fit_score
# paramtry jakich chcemy szukać

def save_prospect(company,url,reason,fit_score):

    # metodą with open otwoeramy plik csv, a oznacza append czyli dopisz,
    # newline zacznie od nowej lini kazdy rekord encoding oznacza polskie znaki
    # f to pojedyczny zapisany obiekt

    with open ('prospect.csv', 'a' , newline='', encoding='utf-8') as f:

        # metodą csv writer,tworzymy obiekt który umie zapisycwać do csv

        writer = csv.writer(f)

        # do pliku metodą writerow dopsiujemy nasze paramtetry company,url,reason,fit_score
        writer.writerow([company,url,reason,fit_score])

        # zwracamy ptwoerzedzie do moedlu ai ze uzylismy narzędzua
        return {'recorded':'ok'}

In [320]:
# opis dla modelu ai o 1 narzędziu ze moze je uzyc u jakiue argyme ty przyjmuje to narzędzie 

scrape_website_json = {
    "name": "scrape_website",
    "description": "Scrape the content of a website to analyze what the company does",
    "parameters": {
        "type": "object",
        "properties": {
            "url": {
                "type": "string",
                "description": "The URL of the website to scrape"
            }
        },
        "required": ["url"],
        "additionalProperties": False
    }
}


In [321]:
search_similar_companies_json = {
    "name": "search_similar_companies",
    "description": "Search the internet for companies similar to the analyzed website",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query to find similar companies"
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}


In [322]:
save_prospect_json = {
    "name": "save_prospect",
    "description": "Save a potential partner company to the prospects list",
    "parameters": {
        "type": "object",
        "properties": {
            "company": {
                "type": "string",
                "description": "Name of the company"
            },
            "url": {
                "type": "string",
                "description": "Website URL of the company"
            },
            "reason": {
                "type": "string",
                "description": "Why this company is a good fit as a partner"
            },
            "fit_score": {
                "type": "number",
                "description": "Score from 1 to 10 indicating how good a fit this company is"
            }
        },
        "required": ["company", "url", "reason", "fit_score"],
        "additionalProperties": False
    }
}


In [323]:

# toold lista narzedzi json dla ai 

tools = [

    {'type' : 'function', "function" : scrape_website_json},
    {'type' : 'function' , "function" : search_similar_companies_json},
    {'type' : 'function' , 'function' : save_prospect_json}
]

In [324]:

# tworzymy funkcje która będzie przujmować nasze tool_calls czyli
# listę narzędzi które stworzylismy model ai moze je wywołac dzieki temu

from unittest import result


def handle_tool_calls(tool_calls):

    # tworzymy tablice do której bedziemy wrzucać wyniki funkcji czyli uzyte narzęedzia
    # to zwraca nasza funkcja uzyte narzędzia przez ai, przyda sie do w dalszej pracy w kolejnych funkcjach
    # dzieki temu model decyuje jakiej funkcji uzyje 

    results = []
    # pętlo for zapętlamy po narzędziach przekazanych w parametrze tool_calls
    # tool_call to pojedyncze narzędzie

    for tool_call in tool_calls:

        # do zmiennej tool_nama przekazujemy wyjęty parametr name z jsona danego tools:
        tool_name = tool_call.function.name
        # do zmiennej arguments wyciągamy arguemnty z kazdego narzędzia
        # które potem przekazemy do wywołanego narzędzia
        # metodą json.loads którą wywołujemy a dalej przekazujemy jako parametr opis z json
        arguments = json.loads(tool_call.function.arguments)

        # teraz robimy warunki, do result przekazujujemy daną funkcje jesli jej 
        # name zostanie uzyty orzez model ai

        if tool_name == 'scrape_website':

            result = scrape_website(**arguments)
        elif tool_name == 'search_similar_companies':

            result = search_similar_companies(**arguments)

        elif tool_name == "save_prospect":

            result = save_prospect(**arguments)


        # teraz do tablicy results appendujemy wynik wywołanego narzędzia
        # zgodnie z openAI api 
        # role = tool content dompujemy metodą json.dump i jako parametr
        # przekazujemy result czyli wywołane nrzędzie w petli if elif z argumentami
        # i jako tool call id przekazjy id danego narzędzia

        results.append({

            'role' : 'tool',
            'content' : json.dumps(result),
            'tool_call_id' : tool_call.id

    }) 
    return results





In [325]:
system_prompt = """Jesteś asystentem do wyszukiwania partnerów dla Last Agency - polskiej agencji SEO/SEM/GEO/AI Search z Poznania.

Dostajesz URL firmy która jest wzorcowym partnerem Last Agency. Twoim zadaniem jest znalezienie firm PODOBNYCH do tej firmy - bo jeśli jedna taka firma jest partnerem, podobne firmy też mogą być partnerami.

Jak działasz:
1. Najpierw użyj scrape_website żeby przeanalizować podany URL
2. Zidentyfikuj profil firmy: branża, usługi, technologie, lokalizacja, typ klientów
3. Wygeneruj 3-4 zapytania szukające PODOBNYCH firm w Polsce - tej samej branży, podobnych usług
4. Użyj search_similar_companies dla każdego zapytania
5. Dla każdej podobnej firmy użyj save_prospect żeby ją zapisać

Zasady:
- Szukaj firm PODOBNYCH do wzorca - ta sama branża, podobne usługi, podobny typ klientów
- Firma musi być z Polski
- Firma NIE może już oferować SEO/SEM/GEO (byłaby konkurentem Last Agency)
- Zapisuj TYLKO bezpośrednie strony firm - nigdy katalogi: clutch.co, sortlist.com, designrush.com, semrush.com, wadline.com
- Nigdy nie zapisuj tej samej firmy dwa razy
- Minimalny fit_score 7"""


In [326]:
evaluator_system_prompt = """Oceniasz czy znalezione firmy są dobrymi potencjalnymi partnerami dla Last Agency - polskiej agencji SEO/SEM/GEO z Poznania oferującej partnerstwa white-label i referral.

Dobry partner:
- Jest polską agencją lub firmą z własnymi klientami biznesowymi
- Oferuje komplementarne usługi: web development, software house, branding, social media, PR, marketing automation
- NIE oferuje SEO/SEM/GEO (nie jest bezpośrednim konkurentem)
- Ma prawdziwy bezpośredni URL strony (nie katalog jak clutch.co, sortlist.com itp.)
- Brak duplikatów firm w wynikach

Odrzuć wyniki jeśli:
- Jakiekolwiek URL prowadzą do katalogów zamiast stron firm
- Są zduplikowane firmy
- Firmy działają poza Polską
- Firmy już oferują SEO/SEM/GEO

Zwróć TYLKO JSON z polami:
- is_acceptable (bool): czy wyniki są wystarczająco dobre
- feedback (string): co było nie tak lub co poprawić jeśli nie są akceptowalne"""


In [327]:

# tworzymy clase która opiera sie o model pydatica BaseModel, model bierze odpowiedz od llm, i zamienia ją w obiekt z 2 wartosciami 1 to boolean
# true lub false // potem dorzyca feedback w stringu  



class Evaluation(BaseModel):
    is_acceptable: bool
    feedback : str

In [328]:

# model evalutor, llm który otrzmuje systemowy prompt zwraca odpowiedz w formacie evaluation 

def evaluate(reply,message,history) -> Evaluation:

    # definiujemy messages które przekazemy do llm'a 

    messages = [

    {'role' : 'system' , "content" : evaluator_system_prompt},
        {'role': 'user', 'content': f"Here are the found prospects:\n{reply}\n\nOriginal URL analyzed:\n{message}"}

    ]


    # piszemy model ale musimy uzyc parse który tworzy obkekt, bo nasza odpowiedz nie bedze tworzona
    # przez string tylko idzie do modelu BaseModel a on jest obiektem

    response = openai.beta.chat.completions.parse(
            model='gpt-4.1-mini',
        messages=messages,
        # dodatkowo zwracamy format odpowiedzi jako objekt
        response_format = Evaluation


    )

    # zwracamy odpowiedz do Evaluation w postaci obiektu metoda parsed

    return response.choices[0].message.parsed


In [329]:
# piszemy funkcje rerun, która jest uruchamiana jesli model eavlaute, zwróci odpowiedz, czyli przenalizuje
# odpoiwedz agenta 1 ta odpowiedz trafi do modelu Evaliatuion i zwróci false, wówczas odpalamy rerun
# wzbogacony o feedback



def rerun(reply,message,history,feedback): 

    # tworzymy system prompt w oparciu o klasyczny system prompt + parametry jak reply, message, feedback

    new_system_prompt = system_prompt + f"\n\n## Previous results rejected\n" # dajemy stary system prompt + info o odrzuceniu
    new_system_prompt +=  f"Your previous results:\n{reply}\n\n" # dajemy mu poprzednią odpowiedz
    new_system_prompt += f"Feedback:\n{feedback}\n\n" # dajemy feedback
    new_system_prompt +=  "Please search again with different queries and find better matching companies."  # prosmy o nowy match

    # tworzymy messages na podstawie nowego system prompty , obiektu history oraz message
    messages = [{"role": "system", "content": new_system_prompt}]  + history +  [{"role": "user", "content": message}]


    # teraz robimy petle while, która działa dopóki flaga nie zamieni sie na true, zamiani sie jak llm
    # zakonczy uzywac narzędzia

    done = False

    while not done: 
    
    # tworzymy model openai, który ma nowe messages i tools czyli liste naszych narzędzi
        response = openai.chat.completions.create(

                model = 'gpt-4.1-mini',
                messages=messages,
                tools=tools
        )

        # tworzymy zmienną finish reason, która sprawdza czy llm bedzie uzywal narzedzia czy skonczył działać bez narzedzi

        finish_reason = response.choices[0].finish_reason

        # sprawdzamy petla if czy finish_reason, zwróci jakies narzędzia tool_calls 
        #jesli model zuzyje narziedzia zawsze do finish_reason zwróci tool_calls
        # a jak zakonczy to zwróci do niish reason stop

        if finish_reason == 'tool_calls':

            #wyciągamy cały message od modelu
            message_obj = response.choices[0].message
            # wyciągamy z message tool_calls  czyli liste izytych narzedzi, ktore stworzylismy
            tool_calls = message_obj.tool_calls
            # wywołujemy funkcje hadne tool calls i jej wynik przekazujemy do zmiennej, a haki poaramtr uzyjemy
            # wyciagnietej listy narzedzi z message
            result = handle_tool_calls(tool_calls)
            # do listy messages dorzucamy cały message z modelu , zeby model wiedział jakie narzędzia zamówiono
            messages.append(message_obj)
            # do listy messages dopisujemy wyniki narzedzi zeby model wiedział co dostał 
            messages.extend(result)

        # jesli model nie chce uzywac narzedzi i finish reason bedzie stop zmieniamy done na True i koniec petli

        else:
            done = True

    return response.choices[0].message.content # zwracamy odpowiedz finalną


In [330]:

# piszemy finalną fukcje chat przyjuje message i history



def chat(message,history) :

    # lista message, system prompt , history, + message

    messages = [{'role' : 'system', 'content' : system_prompt}] + history + [{'role' : 'user'  ,'content'  : message }]


    # teraz pętla while która trwa póki done jest false

    done = False

    while not done :

        # budujemy model

        response = openai.chat.completions.create(

            model = 'gpt-4.1-mini',
            messages=messages,
            tools=tools

        )

        # tworzymy zmienna finish reason kóra zwraca albo stop = bot nie uzywa juz narzedzi
        # albo = tool_calls = uzywa narzedzi

        finish_reason = response.choices[0].finish_reason

        # if else jesli finish reason to tool_calls to:

        if finish_reason == 'tool_calls':

            # wyciągamy z modelu cały message
            message_obj = response.choices[0].message
            # wyciagamy narziędzia z message obj, czyli z modelu
            tool_calls = message_obj.tool_calls
            # uruchamiamy funkcje która uruchamia narzedzia i jako partmetr dajemy konkretne tool_calls
            result = handle_tool_calls(tool_calls)
            # do message, które trafia do modelu dajemy info o obiekcie message
            messages.append(message_obj)
            # do message które trafia do modelu dajemy metodą extend wynik funkcju handle_tool_calls 
            # które znajduje się w result
            messages.extend(result) 

        else:
            done = True

        # zwracamy odpowiedz modelu 
    reply = response.choices[0].message.content

        # teraz uruchamiamy model evaluate z reply które mamy tutaj wytworzone,
        # i przekazujemy je do klasy BaseModel evalaution, któe wezmie nasza odpowiedz i utworzy obiekt 
        # z 2 polami

    evaluation = evaluate(reply,message,history)

    if evaluation.is_acceptable: 
            print("Passed evaluation - returning results")
    else:
            print(f"Failed evaluation - retrying\n{evaluation.feedback}")
            # jesli is_accptable to false uruchamiamy rerun
            reply =  rerun(reply,message,history,evaluation.feedback)

    return reply 




In [334]:
history = []

url = "https://www.tebim.pro/"

reply = chat(url, history)

print(reply)


Passed evaluation - returning results
Analizując firmę Tebim, jest to agencja z Polski specjalizująca się w kompleksowych wdrożeniach sklepów internetowych na platformie PrestaShop, oferująca takie usługi jak: wdrożenia sklepów B2C i B2B, rozwój i utrzymanie platform e-commerce, konsultacje technologiczne, integracje systemów ERP, PIM, CRM, audyty UX/UI oraz dedykowane moduły. Firma działa głównie w branży e-commerce, obsługuje polskich klientów i nie jest konkurentem w zakresie SEO/SEM/GEO.

W oparciu o profil tej firmy wygenerowałem zapytania dotyczące agencji i firm z Polski:

1. agencja ecommerce polska wdrożenia sklepów internetowych Prestashop
2. firma wdrażająca platformy B2B ecommerce Polska
3. agencja projektowanie UX UI sklepy internetowe Polska
4. firma integracje ERP CRM ecommerce Polska

Znalazłem i zapisałem odpowiednie, podobne firmy z Polski, które nie oferują SEO/SEM/GEO, ale mają dopasowany profil usług. Oto przykładowe firmy, które zostały zapisane jako potencjalni p